# Chagas Disease ECG Detection — Baseline Pipeline
**PhysioNet Challenge 2025 | CODE-15% Dataset**

This notebook implements a clean, leak-free baseline:
1. Load ECG signals from converted WFDB files
2. Load ground-truth labels from `code15_chagas_labels.csv`
3. Split → Normalize (train stats only) → Convert to tensors
4. Train a 1D-CNN with `BCEWithLogitsLoss`
5. Evaluate with Accuracy + AUC

**Key rules this notebook follows to avoid common bugs:**
- Labels come from CSV, never from ECG header comments
- Normalization is computed on train set only, applied to both
- `BCEWithLogitsLoss` is used (no `Sigmoid` in the model)
- `torch.sigmoid()` is applied only at prediction time

## 1 — Imports

In [ ]:
import os
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Paths — adjust if your folder structure is different
DATA_PATH   = "../processed_data"   # folder with .dat/.hea files
LABELS_PATH = "../data/code15_chagas_labels.csv"

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("All imports OK")

## 2 — Quick sanity check: plot one ECG

In [ ]:
# Just grab the first .hea file we can find
sample_name = None
for f in os.listdir(DATA_PATH):
    if f.endswith('.hea'):
        sample_name = f.replace('.hea', '')
        break

rec = wfdb.rdrecord(os.path.join(DATA_PATH, sample_name))
sig = rec.p_signal
print(f"Record: {sample_name}  |  Signal shape: {sig.shape}  (timesteps x leads)")

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
lead_names = rec.sig_name if rec.sig_name else [f'Lead {i}' for i in range(12)]
for i, ax in enumerate(axes):
    ax.plot(sig[:2000, i], linewidth=0.8, color='steelblue')
    ax.set_ylabel(lead_names[i], fontsize=9)
    ax.set_yticks([])
axes[-1].set_xlabel('Sample index')
fig.suptitle('Sample ECG — first 3 leads', fontsize=12)
plt.tight_layout()
plt.show()

## 3 — Load dataset with correct labels

Labels come from `code15_chagas_labels.csv`, **not** from ECG header comments.
Using comments caused the all-ones label bug in the early experiments.

In [ ]:
# Build exam_id → chagas label mapping
labels_df  = pd.read_csv(LABELS_PATH)
print("Label CSV columns:", labels_df.columns.tolist())
print(labels_df.head(3))

# The exam_id column might be named differently — inspect and adjust if needed
ID_COL     = labels_df.columns[0]   # first column = exam id
LABEL_COL  = 'chagas'               # adjust if column name differs
label_map  = dict(zip(labels_df[ID_COL].astype(str), labels_df[LABEL_COL]))
print(f"\nTotal labelled exams in CSV: {len(label_map)}")

In [ ]:
N_SAMPLES   = 300    # increase later once pipeline is confirmed working
TARGET_LEN  = 2000   # timesteps per ECG (2000 = 5s at 400Hz)

def fix_length(sig, target=TARGET_LEN):
    """Truncate or zero-pad signal to fixed length."""
    if sig.shape[0] >= target:
        return sig[:target]
    return np.pad(sig, ((0, target - sig.shape[0]), (0, 0)))

all_hea = [f.replace('.hea', '') for f in os.listdir(DATA_PATH) if f.endswith('.hea')]
all_hea = sorted(all_hea)
print(f"Total records on disk: {len(all_hea)}")

X, y = [], []
skipped = 0

for name in all_hea[:N_SAMPLES]:
    try:
        rec = wfdb.rdrecord(os.path.join(DATA_PATH, name))
        sig = rec.p_signal
        if sig is None or sig.shape[1] != 12 or np.any(np.isnan(sig)):
            skipped += 1
            continue
        X.append(fix_length(sig))
        y.append(label_map.get(name, 0))   # default 0 if not in CSV
    except Exception:
        skipped += 1
        continue

X = np.array(X, dtype=np.float32)   # (N, 2000, 12)
y = np.array(y, dtype=np.float32)

unique, counts = np.unique(y, return_counts=True)
print(f"\nLoaded: {len(X)} samples  |  Skipped: {skipped}")
print(f"Label distribution: { {int(u): int(c) for u, c in zip(unique, counts)} }")

## 4 — Split → Normalize → Convert to tensors

**Order matters:**
1. Split first (so test set is truly unseen)
2. Compute normalization stats from train only
3. Apply same stats to test (no leakage)

In [ ]:
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalize using TRAIN stats only
max_val = np.max(np.abs(X_train_np)) + 1e-8
X_train_np = X_train_np / max_val
X_test_np  = X_test_np  / max_val   # same scale, no leakage

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_np)
X_test  = torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np).unsqueeze(1)
y_test  = torch.tensor(y_test_np).unsqueeze(1)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train label dist: 0={int((y_train==0).sum())}  1={int((y_train==1).sum())}")
print(f"Test  label dist: 0={int((y_test==0).sum())}   1={int((y_test==1).sum())}")

## 5 — Model definition

3-layer 1D-CNN with `AdaptiveAvgPool1d` (no hardcoded dimensions).  
**No Sigmoid at the end** — `BCEWithLogitsLoss` handles that internally.

In [ ]:
class ECGBaseline(nn.Module):
    """
    Lightweight 1D-CNN for 12-lead ECG binary classification.
    Input shape:  (batch, timesteps, 12)
    Output shape: (batch, 1)  — raw logit, NO sigmoid
    """
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            # Block 1
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # Block 2
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # Block 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            # Collapse time dimension — works for any input length
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            # NO Sigmoid here — BCEWithLogitsLoss includes it
        )

    def forward(self, x):
        # x: (batch, timesteps, channels) → permute to (batch, channels, timesteps)
        x = x.permute(0, 2, 1)
        return self.classifier(self.encoder(x))


model = ECGBaseline()

# Quick shape test
dummy = torch.zeros(4, TARGET_LEN, 12)
out   = model(dummy)
print(f"Model output shape: {out.shape}  (expected: torch.Size([4, 1]))")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 6 — Training

In [ ]:
EPOCHS = 25
LR     = 1e-3

criterion = nn.BCEWithLogitsLoss()   # numerically stable, handles class imbalance better
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    # Shuffle training data each epoch
    perm    = torch.randperm(X_train.size(0))
    X_shuf  = X_train[perm]
    y_shuf  = y_train[perm]

    logits  = model(X_shuf)
    loss    = criterion(logits, y_shuf)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    train_losses.append(loss.item())

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.5f}")

# Plot loss curve
plt.figure(figsize=(8, 3))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.tight_layout()
plt.show()

## 7 — Evaluation

**Important:** `torch.sigmoid()` is applied here at prediction time.  
It was deliberately excluded from the model so `BCEWithLogitsLoss` could work correctly.

In [ ]:
model.eval()
with torch.no_grad():
    logits      = model(X_test)
    probs       = torch.sigmoid(logits)          # sigmoid ONLY here
    predictions = (probs > 0.5).float()

y_true = y_test.numpy().ravel()
y_prob = probs.numpy().ravel()
y_pred = predictions.numpy().ravel()

accuracy = (y_pred == y_true).mean()
auc      = roc_auc_score(y_true, y_prob)

print(f"{'='*40}")
print(f"  Test Accuracy : {accuracy:.4f}")
print(f"  AUC-ROC       : {auc:.4f}")
print(f"{'='*40}")
print()
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, target_names=['No Chagas', 'Chagas']))

## 8 — Save model

Saves weights so you don't need to retrain every session.

In [ ]:
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/ecg_baseline.pt')
print("Model saved to ../models/ecg_baseline.pt")

# To reload later:
# model = ECGBaseline()
# model.load_state_dict(torch.load('../models/ecg_baseline.pt'))
# model.eval()

## 9 — Next steps

Once this baseline runs cleanly, the project roadmap continues:

| Step | What | Why it matters |
|------|------|----------------|
| Phase 2 | Signal preprocessing (bandpass filter, resample to common Hz) | CODE-15% is 400Hz, PTB-XL is 500Hz — need consistent input |
| Phase 2 | Scale to 1000+ samples | Better generalization estimates |
| Phase 3 | Cross-dataset eval (train CODE-15%, test PTB-XL / SaMi-Trop) | The research gap — tests real-world robustness |
| Phase 3 | GradCAM on 1D signals | Explainability = publishable + hireable |
| Phase 4 | INT8 quantization (torch.quantization) | Required before FPGA deployment |
| Phase 4 | FPGA deployment via HLS4ML or FINN | The zero-competition gap this lab is positioned to fill |
| Phase 5 | HuggingFace Spaces demo | Recruiters and reviewers can interact without running code |